# CS484/684 Final Project (Winter 2026)
# Project 8: Weakly-Supervised Semantic Segmentation

**Student 1:** Andrey (Andrey@swimhomes.ca)
**Student 2:** [Partner Name] ([UW Email])

Three approaches to WSSS on PASCAL VOC 2012, all trained on image-level class labels derived from segmentation masks:

1. **Basic**: frozen DINOv3 ViT-S + linear classifier, CAMs via fc weights
2. **DINO + SAM Distillation**: DINOv3 ViT-L + convolutional decoder + SAM2 pseudo-label distillation
3. **CLIP-ES + DINO + SAM2**: training-free pipeline using CLIP GradCAM, DINO affinity refinement, and SAM2 (CPM prompting)

Each section is self-contained: run its **Setup** cell + **Evaluation** cell to score a saved checkpoint without re-training.

---
## Shared Setup

Common imports and utilities from `dataset.py`. Run this cell before any architecture section.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from dataset import (
    make_voc_datasets,
    VOC_CLASSES,
    colorize,
    denorm_to_uint8,
    get_wsss_dataloaders,
    cache_dino_tokens,
    evaluate_masks,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## Architecture 1 - Basic WSSS (DINOv3 + Linear Classifier)

The simplest approach: a **frozen DINOv3 ViT-Small** backbone with a single linear classification head trained with image-level binary cross-entropy. Class Activation Maps are generated from the classifier weights following [Zhou et al., CVPR 2016](https://arxiv.org/abs/1512.04150), equivalent to a 1×1 convolution of the spatial feature map with the fc weight matrix.

Since DINOv3 is fully frozen, patch tokens are cached once before training and reused every epoch, making training very fast. The trade-off is spatial coarseness: at 224×224 with 16×16 patches the feature grid is only 14×14, so CAMs are blocky. No spatial refinement is applied.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 20
BASIC_CHECKPOINT = "basic_checkpoint.pth"

print("Loading DINOv3 ViT-Small (frozen)...")
dino_model = torch.hub.load(
    "./dinov3",
    "dinov3_vits16",
    source="local",
    weights="weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth",
).to(device)
dino_model.eval()

EMBED_DIM = dino_model.embed_dim  # 384
PATCH_SIZE = dino_model.patch_size  # 16
GRID = IMG_SIZE // PATCH_SIZE  # 14

train_ds, val_ds = make_voc_datasets(root="./data", resize_size=IMG_SIZE)
train_loader, val_loader = get_wsss_dataloaders(
    train_ds, val_ds, batch_size=BATCH_SIZE
)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

train_tokens, train_labels = cache_dino_tokens(
    dino_model, train_ds, train_loader, device=device
)
val_tokens, val_labels = cache_dino_tokens(
    dino_model, val_ds, val_loader, device=device
)

In [ ]:
class CAMClassifier(nn.Module):
    # Linear classifier; CAMs via fc weight convolution (Zhou et al., CVPR 2016)

    def __init__(self, embed_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(embed_dim, num_classes, bias=True)

    def forward(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        return self.fc(patch_tokens.mean(dim=1))

    @torch.no_grad()
    def cam(self, patch_tokens: torch.Tensor, grid: int) -> torch.Tensor:
        B, P, D = patch_tokens.shape
        feats = patch_tokens.transpose(1, 2).reshape(B, D, grid, grid)
        return F.conv2d(feats, self.fc.weight.unsqueeze(-1).unsqueeze(-1))

    @torch.no_grad()
    def predict(self, patch_tokens, grid, target_size, image_labels, bg_threshold=0.25):
        cams = F.relu(self.cam(patch_tokens.float(), grid))
        cams = F.interpolate(
            cams, size=target_size, mode="bilinear", align_corners=False
        )
        B, C, H, W = cams.shape
        flat = cams.view(B, C, -1)
        flat = (flat - flat.min(2, keepdim=True).values) / (
            flat.max(2, keepdim=True).values - flat.min(2, keepdim=True).values
        ).clamp(min=1e-6)
        cams = flat.view_as(cams) * image_labels.view(B, C, 1, 1)
        bg = torch.full(
            (B, 1, H, W), bg_threshold, device=cams.device, dtype=cams.dtype
        )
        return cams, torch.cat([bg, cams], dim=1).argmax(dim=1)

In [ ]:
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4

classifier = CAMClassifier(EMBED_DIM, NUM_CLASSES).to(device)
optimizer = AdamW(classifier.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
bce_loss = nn.BCEWithLogitsLoss()


def cached_batches(tokens, labels, batch_size, shuffle=True):
    n = tokens.shape[0]
    order = (
        torch.randperm(n, device=tokens.device)
        if shuffle
        else torch.arange(n, device=tokens.device)
    )
    for i in range(0, n, batch_size):
        idx = order[i : i + batch_size]
        yield tokens[idx].float(), labels[idx]


epoch_losses = []
epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training epochs", total=EPOCHS, leave=True)
for epoch in epoch_bar:
    classifier.train()
    epoch_loss, n_seen = 0.0, 0
    for x, y in cached_batches(train_tokens, train_labels, BATCH_SIZE):
        optimizer.zero_grad()
        loss = bce_loss(classifier(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
        n_seen += x.size(0)
    scheduler.step()
    avg = epoch_loss / n_seen
    epoch_losses.append(avg)
    epoch_bar.set_postfix(loss=f"{avg:.4f}")

torch.save(
    {
        "epoch": EPOCHS,
        "model_state_dict": classifier.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch_losses": epoch_losses,
    },
    BASIC_CHECKPOINT,
 )
print(f"Checkpoint saved to {BASIC_CHECKPOINT}")

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker="o")
plt.title("Basic WSSS — Training Loss")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.grid(True)
plt.show()

In [ ]:
def visualize_basic(num_images=3, seed=0):
    classifier.eval()
    rng = random.Random(seed)
    indices = rng.sample(range(len(val_ds)), num_images)
    fig, axes = plt.subplots(num_images, 3, figsize=(12, 4 * num_images))
    if num_images == 1:
        axes = [axes]
    for row, idx in zip(axes, indices):
        _, img_t, label_vec, _ = val_ds[idx]
        tokens = val_tokens[idx].unsqueeze(0).float()
        cams, preds = classifier.predict(tokens, GRID, (IMG_SIZE, IMG_SIZE), label_vec.unsqueeze(0).to(device))
        present = [VOC_CLASSES[c] for c in torch.where(label_vec > 0)[0].tolist()]
        row[0].imshow(denorm_to_uint8(img_t)); row[0].set_title(", ".join(present)); row[0].axis("off")
        row[1].imshow(cams[0].max(0).values.cpu().numpy(), cmap="jet", vmin=0, vmax=1)
        row[1].set_title("max CAM"); row[1].axis("off")
        row[2].imshow(colorize(preds[0].cpu().numpy())); row[2].set_title("predicted mask"); row[2].axis("off")
    plt.tight_layout(); plt.show()

visualize_basic()

In [ ]:
# Load checkpoint — run this cell independently of training to evaluate a saved model
ckpt = torch.load(BASIC_CHECKPOINT, map_location=device, weights_only=False)
classifier = CAMClassifier(EMBED_DIM, NUM_CLASSES).to(device)
classifier.load_state_dict(ckpt["model_state_dict"])
classifier.eval()
print(f"Loaded {BASIC_CHECKPOINT} (epoch {ckpt['epoch']})")

n_cls = NUM_CLASSES + 1
inter = np.zeros(n_cls, dtype=np.int64)
union = np.zeros(n_cls, dtype=np.int64)

for indices, images, labels, masks in tqdm(val_loader, desc="mIoU"):
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    tokens = val_tokens[indices].float()
    _, preds = classifier.predict(tokens, GRID, (IMG_SIZE, IMG_SIZE), labels)
    for b in range(len(indices)):
        mask_t = masks[b].to(device)
        H, W = mask_t.shape
        p = F.interpolate(
            preds[b : b + 1].float().unsqueeze(0), (H, W), mode="nearest-exact"
        ).squeeze()
        b_int, b_uni, _ = evaluate_masks(p, mask_t, num_classes=n_cls, ignore_index=255)
        inter += b_int.cpu().numpy().astype(np.int64)
        union += b_uni.cpu().numpy().astype(np.int64)

iou = inter / np.maximum(union, 1)
present = union > 0
miou = iou[present].mean()
print(f"\nmIoU = {miou:.4f}")
for c, name in enumerate(["background"] + VOC_CLASSES):
    if present[c]:
        print(f"  {name:<14s}  {iou[c]:.4f}")

---
## Architecture 2 - DINO + SAM Distillation

A **frozen DINOv3 ViT-Large** backbone feeds patch tokens into a lightweight convolutional decoder (Conv-BN-ReLU + bilinear upsampling) that produces dense class logits at the input resolution. The spatial max-pool gives an image-level score, supervised by BCE on image-level labels.

Spatial supervision comes from **SAM2 pseudo-label distillation**: on each training iteration, the current dense CAMs are used to extract a bounding-box + peak-point prompt per active class, which SAM2 uses to produce a segmentation mask. That mask becomes the per-class training target for a combined BCE + Dice loss. The model bootstraps spatial supervision from its own improving predictions.

A **burn-in** phase (classification loss only) runs for the first epoch to let CAMs stabilise before distillation begins. A checkpoint is saved after every epoch so training can be resumed.

In [ ]:
import torch.optim as optim
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [ ]:
IMG_SIZE = 448
BATCH_SIZE = 16
NUM_CLASSES = 20
DINO_SAM_CHECKPOINT = "dino_sam_checkpoint.pth"

print("Loading DINOv3 ViT-Large (frozen)...")
dino_model = torch.hub.load(
    "./dinov3",
    "dinov3_vitl16",
    source="local",
    weights="weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth",
).to(device)
dino_model.eval()

sam2_model = build_sam2(
    "configs/sam2.1/sam2.1_hiera_s.yaml", "weights/sam2.1_hiera_small.pt", device=device
)
sam_predictor = SAM2ImagePredictor(sam2_model)
print("Models loaded.")

train_ds, val_ds = make_voc_datasets(root="./data", resize_size=IMG_SIZE)
train_loader, val_loader = get_wsss_dataloaders(
    train_ds, val_ds, batch_size=BATCH_SIZE
)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

dino_cache, train_labels = cache_dino_tokens(
    dino_model, train_ds, train_loader, device=device
)

In [ ]:
class UnifiedWSSSSegmenter(nn.Module):
    def __init__(self, in_channels=1024, num_classes=20):
        super().__init__()
        self.in_channels = in_channels
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(512, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(128, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, kernel_size=1),
        )

    def forward(self, patch_tokens, grid_size, target_size):
        x = patch_tokens.permute(0, 2, 1).reshape(-1, self.in_channels, *grid_size)
        logits = self.decoder(x)
        dense_logits = F.interpolate(
            logits, size=target_size, mode="bilinear", align_corners=False
        )
        return dense_logits.amax(dim=(2, 3)), dense_logits

    @torch.no_grad()
    def predict(self, patch_tokens, grid_size, target_size, bg_threshold=0.5):
        _, dense_logits = self.forward(patch_tokens, grid_size, target_size)
        probs = torch.sigmoid(dense_logits)
        B, _, H, W = probs.shape
        bg = torch.full((B, 1, H, W), bg_threshold, device=probs.device)
        return torch.cat([bg, probs], dim=1).argmax(dim=1)

In [ ]:
NUM_EPOCHS = 30
BURN_IN_EPOCHS = 10
LEARNING_RATE = 1e-4
CAM_THRESHOLD = 0.4


def seg_bce_dice_loss(preds, targets, dice_weight=0.5, dice_smoothing=1.0):
    bce = F.binary_cross_entropy_with_logits(preds, targets)
    probs = torch.sigmoid(preds)
    intersection = (probs * targets).sum()
    union = probs.sum() + targets.sum()
    dice = 1 - (2.0 * intersection + dice_smoothing) / (union + dice_smoothing)
    return bce + dice_weight * dice


@torch.no_grad()
def generate_sam_pseudo_labels(
    images, dense_logits, image_level_labels, cam_threshold=0.4
):
    B, _, H, W = images.shape
    pseudo = torch.zeros((B, NUM_CLASSES, H, W), dtype=torch.float32, device=device)
    for b in range(B):
        active = torch.where(image_level_labels[b] > 0)[0]
        if len(active) == 0:
            continue
        sam_predictor.set_image(denorm_to_uint8(images[b]))
        C = len(active)
        cam_flat = dense_logits[b, active].detach().view(C, -1)
        cam_min = cam_flat.min(1, keepdim=True).values
        cam_max = cam_flat.max(1, keepdim=True).values
        cam_norm = (cam_flat - cam_min) / (cam_max - cam_min + 1e-8)
        cam_2d = cam_norm.view(C, H, W)
        max_flat = cam_norm.argmax(1)
        max_y = torch.div(max_flat, W, rounding_mode="trunc")
        max_x = max_flat % W
        mask_bool = cam_2d > cam_threshold
        y_idx = torch.arange(H, device=device).view(1, H, 1).expand(C, H, W)
        x_idx = torch.arange(W, device=device).view(1, 1, W).expand(C, H, W)
        y_min = torch.where(mask_bool, y_idx, H).amin((1, 2)).clamp(0, H)
        y_max = torch.where(mask_bool, y_idx, -1).amax((1, 2)).clamp(0, H)
        x_min = torch.where(mask_bool, x_idx, W).amin((1, 2)).clamp(0, W)
        x_max = torch.where(mask_bool, x_idx, -1).amax((1, 2)).clamp(0, W)
        valid_box = y_min <= y_max
        for c in range(C):
            pt = np.array([[max_x[c].item(), max_y[c].item()]], dtype=np.float32)
            lbl = np.array([1], dtype=np.int32)
            box = (
                np.array(
                    [
                        x_min[c].item(),
                        y_min[c].item(),
                        x_max[c].item(),
                        y_max[c].item(),
                    ],
                    dtype=np.float32,
                )
                if valid_box[c]
                else None
            )
            masks, _, _ = sam_predictor.predict(
                point_coords=pt, point_labels=lbl, box=box, multimask_output=False
            )
            pseudo[b, active[c]] = torch.tensor(masks).squeeze().to(device).float()
    return pseudo


def train_one_epoch(epoch, model, optimizer, is_burn_in):
    model.train()
    total_loss, n_batches = 0.0, 0
    tag = "(burn-in)" if is_burn_in else ""
    for indices, images, image_level_labels, _ in tqdm(
        train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} {tag}", leave=False
    ):
        images = images.to(device, non_blocking=True)
        image_level_labels = image_level_labels.to(device, non_blocking=True)
        B, _, H, W = images.shape
        optimizer.zero_grad(set_to_none=True)
        patch_tokens = dino_cache[indices].float()
        grid_dim = int(patch_tokens.shape[1] ** 0.5)
        img_logits, dense_logits = model(patch_tokens, (grid_dim, grid_dim), (H, W))
        loss = nn.BCEWithLogitsLoss()(img_logits, image_level_labels)
        if not is_burn_in:
            pseudo = generate_sam_pseudo_labels(
                images, dense_logits, image_level_labels, CAM_THRESHOLD
            )
            b_idx, c_idx = torch.where(image_level_labels > 0)
            if len(b_idx) > 0:
                loss += seg_bce_dice_loss(
                    dense_logits[b_idx, c_idx], pseudo[b_idx, c_idx]
                )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


segmenter = UnifiedWSSSSegmenter(num_classes=NUM_CLASSES).to(device)
optimizer = optim.Adam(segmenter.parameters(), lr=LEARNING_RATE)
start_epoch, epoch_losses = 0, []

if os.path.exists(DINO_SAM_CHECKPOINT):
    print(f"Loading checkpoint from {DINO_SAM_CHECKPOINT}...")
    ckpt = torch.load(DINO_SAM_CHECKPOINT, map_location=device, weights_only=False)
    segmenter.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"]
    epoch_losses = ckpt.get("epoch_losses", [])
    print(f"Resuming from epoch {start_epoch}")

epoch_bar = tqdm(
    range(start_epoch, NUM_EPOCHS),
    desc="Training epochs",
    total=NUM_EPOCHS - start_epoch,
    leave=True,
 )
for epoch in epoch_bar:
    avg_loss = train_one_epoch(epoch, segmenter, optimizer, epoch < BURN_IN_EPOCHS)
    epoch_losses.append(avg_loss)
    epoch_bar.set_postfix(loss=f"{avg_loss:.4f}")
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": segmenter.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch_losses": epoch_losses,
        },
        DINO_SAM_CHECKPOINT,
    )

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker="o")
plt.title("DINO + SAM Distillation — Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

In [ ]:
def visualize_dino_sam(num_images=3, seed=0):
    segmenter.eval()
    rng = random.Random(seed)
    indices = rng.sample(range(len(val_ds)), num_images)
    fig, axes = plt.subplots(num_images, 3, figsize=(12, 4 * num_images))
    if num_images == 1:
        axes = [axes]
    for row, idx in zip(axes, indices):
        _, img_t, label_vec, _ = val_ds[idx]
        with torch.no_grad():
            feats = dino_model.forward_features(img_t.unsqueeze(0).to(device))
            patch_tokens = feats["x_norm_patchtokens"].float()
            grid_dim = int(patch_tokens.shape[1] ** 0.5)
            _, dense_logits = segmenter(
                patch_tokens, (grid_dim, grid_dim), (img_t.shape[1], img_t.shape[2])
            )
            pred_probs = torch.sigmoid(dense_logits[0]).cpu()
        active = torch.where(label_vec > 0)[0]
        c_idx = active[0].item() if len(active) > 0 else 0
        cam = pred_probs[c_idx].numpy()
        img_vis = denorm_to_uint8(img_t)
        present = [VOC_CLASSES[c] for c in active.tolist()]
        row[0].imshow(img_vis)
        row[0].set_title(", ".join(present))
        row[0].axis("off")
        row[1].imshow(img_vis)
        row[1].imshow(cam, cmap="jet", alpha=0.5)
        row[1].set_title(f"CAM ({VOC_CLASSES[c_idx]})")
        row[1].axis("off")
        row[2].imshow(img_vis)
        row[2].imshow((cam > 0.5).astype(float), cmap="jet", alpha=0.5)
        row[2].set_title("mask (thr=0.5)")
        row[2].axis("off")
    plt.tight_layout()
    plt.show()


visualize_dino_sam()

In [ ]:
# Load checkpoint — run independently of training
ckpt = torch.load(DINO_SAM_CHECKPOINT, map_location=device, weights_only=False)
segmenter = UnifiedWSSSSegmenter(num_classes=NUM_CLASSES).to(device)
segmenter.load_state_dict(ckpt["model_state_dict"])
segmenter.eval()
dino_model.eval()
print(f"Loaded {DINO_SAM_CHECKPOINT} (epoch {ckpt['epoch']})")

intersection = torch.zeros(21, device=device)
union = torch.zeros(21, device=device)

with torch.inference_mode():
    for indices, images, labels, masks in tqdm(val_loader, desc="mIoU"):
        images = images.to(device, non_blocking=True)
        feats = dino_model.forward_features(images)
        patch_tokens = feats["x_norm_patchtokens"].float()
        grid_dim = int(patch_tokens.shape[1] ** 0.5)
        preds = segmenter.predict(
            patch_tokens, (grid_dim, grid_dim), (images.shape[2], images.shape[3])
        )
        for b in range(len(masks)):
            mask_t = masks[b].to(device)
            H, W = mask_t.shape
            p = F.interpolate(
                preds[b : b + 1].float().unsqueeze(0), (H, W), mode="nearest-exact"
            ).squeeze()
            b_int, b_uni, _ = evaluate_masks(
                p, mask_t, num_classes=21, ignore_index=255
            )
            intersection += b_int
            union += b_uni

iou = intersection / union.clamp(min=1e-8)
present = union > 0
miou = iou[present].mean().item()
print(f"\nmIoU = {miou:.4f}")
for c, name in enumerate(["background"] + VOC_CLASSES):
    if present[c]:
        print(f"  {name:<14s}  {iou[c].item():.4f}")

---
## Architecture 3 - CLIP-ES + DINO Affinity + SAM2 (Training-Free)

The intuition behind this pipeline is that
- CLIP works best when the object is centered in the image. CLIP-ES, the variant we use, partially addresses this limitation, but the resulting maps are still fuzzy and not very well aligned to object boundaries.
- DINOv3 is good at finding objects within an image, and its features give cleaner edges than CLIP-ES.
- By producing Class Activation Maps (CAMs) from both CLIP-ES and DINOv3 and combining them, we hope to see an improvement in mIoU.
- To combine the CLIP-ES output with the DINOv3 output, we use patch affinities, which prior work has shown to be effective. A patch is a small region of the image, and patch affinity is the pairwise cosine similarity between patch features. The idea is that if both DINOv3 and CLIP-ES agree on a region, the resulting CAM should be more accurate.
- Finally, a SAM2 block is added at the end of the pipeline to refine the edges of the DINOv3 + CLIP-ES output.

Unlike Architectures 1 and 2, this pipeline requires no training -- all three models are used purely for inference.

In [ ]:
import sys
import cv2
from pathlib import Path
from PIL import Image
from lxml import etree
from matplotlib.patches import Patch

CLIP_ES_DIR = os.path.abspath("clip_es")
if CLIP_ES_DIR not in sys.path:
    sys.path.insert(0, CLIP_ES_DIR)

import clip
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import scale_cam_image
from clip_text import BACKGROUND_CATEGORY, class_names, new_class_names
from utils import parse_xml_to_dict, scoremap2bbox
import generate_cams_voc12 as ces
from generate_cams_voc12 import (
    zeroshot_classifier,
    reshape_transform,
    ClipOutputTarget,
    _transform_resize,
)

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from cpm import cpm_from_cams
from dataset import make_transform, VOC_PALETTE

### Step 1: CLIP-ES CAMs

The first stage of our pipeline is [CLIP-ES (CVPR 2023)](https://github.com/linyq2117/CLIP-ES).

CLIP-ES uses a frozen CLIP ViT-B/16 with Grad-CAM hooked into the last attention block. This turns image-level labels into CAMs, which are then refined by class-aware attention affinity (CAA) using the model's self attention.

The main advantage of CLIP here is that it doesn't require the object to be centered. Given a text prompt for a class label, it can pick out the regions of the image that the prompt activates.

As for the CLIP-ES implementation itself, we reuse many pieces of code from the CLIP-ES repository. This will be noted in comments in the code below.

In [ ]:
clip_device = "cuda" if torch.cuda.is_available() else "cpu"

CLIP_MODEL_NAME = "ViT-B/16"
PROMPT_TEMPLATES = ["a clean origami {}."]  # empirically best prompt from CLIP-ES

print(f"Loading CLIP-ES on {CLIP_MODEL_NAME}...")
clip_model, _ = clip.load(CLIP_MODEL_NAME, device=clip_device)
clip_model.eval()
ces.device = clip_device

fg_text_features = zeroshot_classifier(new_class_names, PROMPT_TEMPLATES, clip_model)
bg_text_features = zeroshot_classifier(
    BACKGROUND_CATEGORY, PROMPT_TEMPLATES, clip_model
)

cam_extractor = GradCAM(
    model=clip_model,
    target_layers=[clip_model.visual.transformer.resblocks[-1].ln_1],
    reshape_transform=reshape_transform,
)
print("CLIP-ES ready.")
print("Foreground classes:", new_class_names)
print("Background classes:", BACKGROUND_CATEGORY)

In [ ]:
VOC_ROOT = "data/VOCdevkit/VOC2012"
PATCH = 16


def run_clip_es_on_image(image_id):
    img_path = os.path.join(VOC_ROOT, "JPEGImages", f"{image_id}.jpg")
    xml_path = os.path.join(VOC_ROOT, "Annotations", f"{image_id}.xml")
    with open(xml_path) as f:
        data = parse_xml_to_dict(etree.fromstring(f.read()))["annotation"]
    ori_W, ori_H = int(data["size"]["width"]), int(data["size"]["height"])
    present_names = []
    for obj in data["object"]:
        if obj["name"] not in present_names:
            present_names.append(obj["name"])
    label_id_list = [class_names.index(n) for n in present_names]
    if not label_id_list:
        raise RuntimeError(f"{image_id} has no labeled objects")

    h = int(np.ceil(ori_H / PATCH) * PATCH)
    w = int(np.ceil(ori_W / PATCH) * PATCH)
    image = _transform_resize(h, w)(Image.open(img_path)).unsqueeze(0).to(clip_device)
    image_features, attn_weight_list = clip_model.encode_image(image, h, w)

    fg_temp = fg_text_features[label_id_list]
    text_temp = torch.cat([fg_temp, bg_text_features], dim=0).to(clip_device)
    cam_input = [image_features, text_temp, h, w]

    coarse_cams, refined_cams = {}, {}
    for col_idx, class_idx in enumerate(label_id_list):
        grayscale_cam, _logits, attn_weight_last = cam_extractor(
            input_tensor=cam_input,
            targets=[ClipOutputTarget(col_idx)],
            target_size=None,
        )
        grayscale_cam = grayscale_cam[0]
        coarse_cams[class_idx] = cv2.resize(grayscale_cam, (ori_W, ori_H))

        attn = torch.stack(
            [a[:, 1:, 1:] for a in (attn_weight_list + [attn_weight_last])], dim=0
        )[-8:]
        attn = attn.mean(dim=0)[0].detach().float().cpu()

        boxes, n_box = scoremap2bbox(
            grayscale_cam, threshold=0.4, multi_contour_eval=True
        )
        aff_mask = torch.zeros(grayscale_cam.shape)
        for i in range(n_box):
            x0, y0, x1, y1 = boxes[i]
            aff_mask[y0:y1, x0:x1] = 1
        aff_mask = aff_mask.view(1, -1)

        trans = attn / attn.sum(dim=0, keepdim=True)
        trans = trans / trans.sum(dim=1, keepdim=True)
        for _ in range(2):
            trans = trans / trans.sum(dim=0, keepdim=True)
            trans = trans / trans.sum(dim=1, keepdim=True)
        trans = (trans + trans.T) / 2
        trans = trans @ trans
        trans = trans * aff_mask
        refined = trans @ torch.tensor(grayscale_cam).view(-1, 1)
        refined_cams[class_idx] = scale_cam_image(
            [refined.view(h // PATCH, w // PATCH).numpy().astype(np.float32)],
            (ori_W, ori_H),
        )[0]

    n_fg = len(fg_temp)
    bg_cam_per_cat = {}
    for bg_local_idx in range(len(bg_text_features)):
        grayscale_cam, _, _ = cam_extractor(
            input_tensor=cam_input,
            targets=[ClipOutputTarget(n_fg + bg_local_idx)],
            target_size=None,
        )
        bg_cam_per_cat[BACKGROUND_CATEGORY[bg_local_idx]] = cv2.resize(
            grayscale_cam[0], (ori_W, ori_H)
        )
    bg_cam = np.maximum.reduce(list(bg_cam_per_cat.values()))
    if bg_cam.max() > 0:
        bg_cam = bg_cam / bg_cam.max()

    return {
        "image_path": img_path,
        "image_size": (ori_W, ori_H),
        "labels": label_id_list,
        "coarse": coarse_cams,
        "refined": refined_cams,
        "bg_cam": bg_cam.astype(np.float32),
    }


def show_cams(result, alpha=0.55):
    img = np.array(Image.open(result["image_path"]).convert("RGB"))
    n = len(result["labels"])
    fig, axes = plt.subplots(2, n + 1, figsize=(4 * (n + 1), 8))
    if n == 1:
        axes = axes.reshape(2, 2)
    for r, (key, label) in enumerate(
        [("coarse", "Grad-CAM"), ("refined", "CAA-refined")]
    ):
        axes[r, 0].imshow(img)
        axes[r, 0].set_title("input")
        axes[r, 0].axis("off")
        for i, c in enumerate(result["labels"]):
            heat = cv2.cvtColor(
                cv2.applyColorMap(
                    (result[key][c] * 255).astype(np.uint8), cv2.COLORMAP_JET
                ),
                cv2.COLOR_BGR2RGB,
            )
            axes[r, i + 1].imshow(
                (alpha * heat + (1 - alpha) * img).clip(0, 255).astype(np.uint8)
            )
            axes[r, i + 1].set_title(f"{label}: {new_class_names[c].split(' ')[0]}")
            axes[r, i + 1].axis("off")
    plt.tight_layout()
    plt.show()


sample_id = "2007_000032"
print(f"Running CLIP-ES on {sample_id}...")
result = run_clip_es_on_image(sample_id)
show_cams(result)

### Step 2: DINO Affinity Refinement

This step uses DINOv3 features to refine the CLIP-ES heatmap from Step 1 via patch affinities.

Patch affinities assign a similarity score to every pair of patches in the image. The idea is to spread the existing CLIP-ES CAM along directions where DINOv3 features agree, and dampen it where they don't, which helps the CAM stick to object boundaries.

This affinity-based refinement was popularized by [Ahn & Kwak, CVPR 2018](https://arxiv.org/abs/1803.10464) ([code](https://github.com/jiwoon-ahn/psa)), and our implementation is based on theirs. The same idea was later adopted in the [MCTformer, CVPR 2022](https://arxiv.org/abs/2203.02891) ([code](https://github.com/xulianuwa/MCTformer)). The novel part here is using DINOv3 patch tokens, rather than a trained affinity network, as the source of the affinities.

In [ ]:
PATCH_DINO = 16

print("Loading DINOv3 ViT-Small (frozen)...")
dino_model = torch.hub.load(
    "./dinov3",
    "dinov3_vits16",
    source="local",
    weights="weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth",
).to(device)
dino_model.eval()
print("DINO loaded.")

In [ ]:
@torch.no_grad()
def dino_patch_affinity(image_path, h, w, threshold=0.2):
    x = make_transform((h, w))(Image.open(image_path)).unsqueeze(0).to(device)
    tokens = F.normalize(
        dino_model.forward_features(x)["x_norm_patchtokens"][0].float(), dim=-1
    )
    affinity = (tokens @ tokens.T - threshold).clamp(min=0)
    return affinity / (affinity.sum(dim=1, keepdim=True) + 1e-8)


@torch.no_grad()
def dino_refine(result, cam_threshold=0.5, n_iter=2):
    ori_W, ori_H = result["image_size"]
    h = int(np.ceil(ori_H / PATCH_DINO) * PATCH_DINO)
    w = int(np.ceil(ori_W / PATCH_DINO) * PATCH_DINO)
    gh, gw = h // PATCH_DINO, w // PATCH_DINO
    N = gh * gw
    affinity = dino_patch_affinity(result["image_path"], h, w).float()
    refined_cams = {}
    for c, cam_full in result["refined"].items():
        cam_grid = cv2.resize(cam_full.astype(np.float32), (gw, gh))
        cam_t = torch.from_numpy(cam_grid).to(device).view(N, 1)
        cam_norm = (cam_grid - cam_grid.min()) / (
            cam_grid.max() - cam_grid.min() + 1e-8
        )
        aff_mask_bool = cam_norm > cam_threshold
        if not aff_mask_bool.any():
            aff_mask_bool = cam_norm >= cam_norm.max()
        aff_mask = (
            torch.from_numpy(aff_mask_bool.astype(np.float32)).to(device).view(1, N)
        )
        trans = affinity * aff_mask
        for _ in range(n_iter + 1):
            trans = trans / (trans.sum(dim=0, keepdim=True) + 1e-8)
            trans = trans / (trans.sum(dim=1, keepdim=True) + 1e-8)
        trans = (trans + trans.T) / 2
        trans = trans @ trans
        refined_cams[c] = scale_cam_image(
            [(trans @ cam_t).view(gh, gw).cpu().numpy().astype(np.float32)],
            (ori_W, ori_H),
        )[0]
    return {**result, "dino_refined": refined_cams}


def show_cams_with_dino(result, alpha=0.55):
    img = np.array(Image.open(result["image_path"]).convert("RGB"))
    rows = [
        ("coarse", "Grad-CAM"),
        ("refined", "CAA-refined"),
        ("dino_refined", "DINO-refined"),
    ]
    n = len(result["labels"])
    fig, axes = plt.subplots(len(rows), n + 1, figsize=(4 * (n + 1), 4 * len(rows)))
    for r, (key, label) in enumerate(rows):
        axes[r, 0].imshow(img)
        axes[r, 0].set_title("input")
        axes[r, 0].axis("off")
        for i, c in enumerate(result["labels"]):
            heat = cv2.cvtColor(
                cv2.applyColorMap(
                    (result[key][c] * 255).astype(np.uint8), cv2.COLORMAP_JET
                ),
                cv2.COLOR_BGR2RGB,
            )
            axes[r, i + 1].imshow(
                (alpha * heat + (1 - alpha) * img).clip(0, 255).astype(np.uint8)
            )
            axes[r, i + 1].set_title(f"{label}: {new_class_names[c].split(' ')[0]}")
            axes[r, i + 1].axis("off")
    plt.tight_layout()
    plt.show()


result_with_dino = dino_refine(result)
show_cams_with_dino(result_with_dino)

From this spot check, we can see that the CAMs are being refined. The edges are still a bit rough, but this was as far as we got. Better ways of integrating CLIP-ES with DINO would be a good direction for future work.

### Step 3: SAM2 Mask Refinement via CPM

To use SAM2, we have to feed it some point prompts (seeds) from the image. Given good seeds, SAM2 should produce a clean segmentation of the object.

To generate these seeds from our CAMs, we follow the method from the [IEEE 2024 paper S2C](https://ieeexplore.ieee.org/document/10658447) ([code](https://github.com/sangrockEG/S2C)).

Specifically, we use their "CPM" block, which "extracts prompts from the CAM of each class and uses them to generate class-specific segmentation masks through SAM". It picks point prompts from the CAM by taking the strongest peak plus a few additional well-separated seeds.

In [ ]:
print("Loading SAM2 Hiera-Large...")
sam_model = build_sam2(
    "configs/sam2.1/sam2.1_hiera_l.yaml",
    "weights/sam2.1_hiera_large.pt",
    device=str(device),
)
sam_predictor = SAM2ImagePredictor(sam_model)
print("SAM2 ready.")

In [ ]:
NUM_FG_CLASSES = len(VOC_CLASSES)


def sam_from_cams(
    result, source="dino_refined", th_multi=0.5, min_distance=20, idx_max_sam=2
):
    cpm_out = cpm_from_cams(
        image=result["image_path"],
        cams=result[source],
        predictor=sam_predictor,
        num_classes=NUM_FG_CLASSES,
        th_multi=th_multi,
        min_distance=min_distance,
        idx_max_sam=idx_max_sam,
    )
    return {**result, "sam_source": source, "sam_masks": cpm_out.masks, "cpm": cpm_out}


def show_cpm(result, alpha=0.55):
    img = np.array(Image.open(result["image_path"]).convert("RGB"))
    cpm_out = result["cpm"]
    n = len(result["labels"])
    fig, axes = plt.subplots(2, n + 1, figsize=(4 * (n + 1), 8))
    axes[0, 0].imshow(img)
    axes[0, 0].set_title("input")
    axes[0, 0].axis("off")
    pgt = cpm_out.pgt
    pgt_rgb = np.zeros((*pgt.shape, 3), dtype=np.uint8)
    palette = (np.array(plt.get_cmap("tab20").colors) * 255).astype(np.uint8)
    for c in result["labels"]:
        pgt_rgb[pgt == c] = palette[c % len(palette)]
    axes[1, 0].imshow(
        (alpha * pgt_rgb + (1 - alpha) * img).clip(0, 255).astype(np.uint8)
    )
    axes[1, 0].set_title("CPM pseudo-label")
    axes[1, 0].axis("off")
    for i, c in enumerate(result["labels"]):
        mask_rgb = np.zeros_like(img)
        mask_rgb[cpm_out.masks[c]] = (255, 64, 64)
        axes[0, i + 1].imshow(img)
        axes[0, i + 1].set_title(new_class_names[c].split(" ")[0])
        axes[0, i + 1].axis("off")
        axes[1, i + 1].imshow(
            (0.55 * mask_rgb + 0.45 * img).clip(0, 255).astype(np.uint8)
        )
        axes[1, i + 1].scatter(
            cpm_out.points[c][:, 0],
            cpm_out.points[c][:, 1],
            s=80,
            marker="*",
            c="lime",
            edgecolors="black",
            linewidths=0.8,
        )
        axes[1, i + 1].set_title(f"SAM (conf={cpm_out.confs[c]:.2f})")
        axes[1, i + 1].axis("off")
    plt.tight_layout()
    plt.show()


print(f"Running CPM + SAM2 on {sample_id}...")
result_with_sam = sam_from_cams(result_with_dino)
show_cpm(result_with_sam)

### Step 4: mIoU Evaluation on VOC 2012 Val

We partly reuse the CLIP-ES repository evaluation code for VOC 2012. We adapt that evaluation loop to our pipeline and also save per-image masks and comparison plots.

In [ ]:
VAL_LIST_PATH = os.path.join(CLIP_ES_DIR, "voc12", "val.txt")
SEG_GT_DIR = os.path.join(VOC_ROOT, "SegmentationClass")
N_CLASS = 21
SOURCES = ["refined", "dino_refined"]
CAM_EVAL_THRES = 2.0
SAVE_DIR = "results"


def save_pred_png(pred, path):
    im = Image.fromarray(pred.astype(np.uint8), mode="P")
    im.putpalette(VOC_PALETTE.tobytes())
    im.save(path)


def _class_legend(ax, mask):
    present = [int(v) for v in np.unique(mask) if 1 <= int(v) <= len(VOC_CLASSES)]
    if not present:
        return
    handles = [
        Patch(color=VOC_PALETTE[c] / 255.0, label=VOC_CLASSES[c - 1]) for c in present
    ]
    ax.legend(
        handles=handles,
        loc="lower right",
        fontsize=8,
        framealpha=0.85,
        handlelength=1.0,
        handleheight=1.0,
    )


def save_comparison_plot(
    image_path, gt, preds_by_source, save_path, ious_by_source=None, alpha=0.55
):
    img = np.array(Image.open(image_path).convert("RGB"))
    ious_by_source = ious_by_source or {}

    def overlay(mask):
        rgb = VOC_PALETTE[np.where(mask == 255, 0, mask)]
        return (alpha * rgb + (1 - alpha) * img).clip(0, 255).astype(np.uint8)

    n_cols = 2 + len(preds_by_source)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    axes[0].imshow(img)
    axes[0].set_title("input")
    axes[0].axis("off")
    axes[1].imshow(overlay(gt))
    axes[1].set_title("ground truth")
    axes[1].axis("off")
    _class_legend(axes[1], gt)
    for i, (source, pred) in enumerate(preds_by_source.items()):
        ax = axes[2 + i]
        ax.imshow(overlay(pred))
        title = f"{source}+SAM"
        if source in ious_by_source:
            title += f"\nmIoU={ious_by_source[source]:.3f}"
        ax.set_title(title)
        ax.axis("off")
        _class_legend(ax, pred)
    plt.tight_layout()
    fig.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)


def load_val_ids(subset=None):
    ids = [ln.strip() for ln in open(VAL_LIST_PATH) if ln.strip()]
    return ids[:subset] if subset else ids


def predict_per_image(result, source, cam_eval_thres):
    keys = np.array(result["labels"], dtype=np.int64)
    cams = np.stack([result[source][c] for c in keys]).astype(np.float32)
    masks = np.stack([result["sam_masks"][c] for c in keys]).astype(np.float32)
    cams = cams * masks
    if cam_eval_thres < 1:
        cams = np.pad(
            cams,
            ((1, 0), (0, 0), (0, 0)),
            mode="constant",
            constant_values=cam_eval_thres,
        )
    else:
        bg = np.power(1.0 - cams.max(axis=0, keepdims=True), cam_eval_thres)
        cams = np.concatenate([bg, cams], axis=0)
    keys_full = np.pad(keys + 1, (1, 0), mode="constant")
    return keys_full[cams.argmax(axis=0)].astype(np.uint8)


def evaluate(
    val_ids,
    sources=SOURCES,
    cam_eval_thres=CAM_EVAL_THRES,
    sam_seed_source="dino_refined",
    save_dir=None,
):
    inter = {s: np.zeros(N_CLASS, dtype=np.int64) for s in sources}
    union_d = {s: np.zeros(N_CLASS, dtype=np.int64) for s in sources}
    skipped = []
    if save_dir is not None:
        save_dir = Path(save_dir)
        for s in sources:
            (save_dir / "masks" / s).mkdir(parents=True, exist_ok=True)
        (save_dir / "plots").mkdir(parents=True, exist_ok=True)
    for image_id in tqdm(val_ids, desc="Evaluating"):
        try:
            res = run_clip_es_on_image(image_id)
        except Exception as e:
            skipped.append((image_id, repr(e)))
            continue
        res = dino_refine(res)
        res = sam_from_cams(res, source=sam_seed_source)
        gt = np.array(Image.open(os.path.join(SEG_GT_DIR, f"{image_id}.png"))).astype(
            np.uint8
        )
        gt_t = torch.from_numpy(gt.astype(np.int64))
        preds_by_source, ious_by_source = {}, {}
        for source in sources:
            pred = predict_per_image(res, source, cam_eval_thres)
            pred_t = torch.from_numpy(pred.astype(np.int64))
            b_int, b_uni, miou_img = evaluate_masks(
                pred_t, gt_t, num_classes=N_CLASS, ignore_index=255
            )
            inter[source] += b_int.numpy().astype(np.int64)
            union_d[source] += b_uni.numpy().astype(np.int64)
            preds_by_source[source] = pred
            ious_by_source[source] = miou_img
            if save_dir is not None:
                save_pred_png(pred, save_dir / "masks" / source / f"{image_id}.png")
        if save_dir is not None:
            save_comparison_plot(
                res["image_path"],
                gt,
                preds_by_source,
                save_dir / f"plots/{image_id}.png",
                ious_by_source,
            )
    if skipped:
        print(f"\nSkipped {len(skipped)}: {skipped[:3]}")
    results = {}
    for s in sources:
        iou = inter[s] / np.maximum(union_d[s], 1)
        present = union_d[s] > 0
        results[s] = {
            "iou": iou,
            "miou": float(iou[present].mean()),
            "present": present,
        }
    return results


def print_results_table(results):
    sources = list(results.keys())
    headers = [f"{s}+SAM" for s in sources]
    name_w = max(len(n) for n in (["background"] + VOC_CLASSES))
    print(f"\n=== mIoU report (cam_eval_thres={CAM_EVAL_THRES}) ===")
    print("class".ljust(name_w) + " | " + " | ".join(h.center(14) for h in headers))
    print("-" * (name_w + 3 + 17 * len(headers)))
    for c, name in enumerate(["background"] + VOC_CLASSES):
        row = [f"{results[s]['iou'][c]:.4f}".center(14) for s in sources]
        print(name.ljust(name_w) + " | " + " | ".join(row))
    print("-" * (name_w + 3 + 17 * len(headers)))
    for s in sources:
        print(f"  mIoU ({s}+SAM): {results[s]['miou']:.4f}")


val_ids = load_val_ids()
print(f"Evaluating on {len(val_ids)} val images...")
results = evaluate(val_ids, save_dir=SAVE_DIR)
print_results_table(results)

### Evaluation Notes

The evaluation is rather disappointing. While we expected the DINO refined pipeline to improve upon the CLIP-ES results, in most cases it ended up working against it, with the mIoU of the DINO-free pipeline largely outperforming the DINO pipeline.

That being said, the results are nonetheless better than the base case pipeline. This is expected due to the stronger capabilities of CLIP-ES and SAM.

Further steps with this pipeline would be to refine and review approaches of combining DINO with CLIP-ES.